# Extractor de Datos SIPA

In [ ]:
import os
import re
import numpy as np
import pdfplumber
from rapidocr_onnxruntime import RapidOCR
import pandas as pd

OCR_LANGUAGE = "es"
OCR_RESOLUTION = 400
OCR_ENGINE = RapidOCR()

MESES = {
    "ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6,
    "JULIO": 7, "AGOSTO": 8, "SEPTIEMBRE": 9, "OCTUBRE": 10,
    "NOVIEMBRE": 11, "DICIEMBRE": 12
}

PROVINCIAS = ["AZUAY", "GUAYAS", "PICHINCHA"]

BBOX_PERECEDEROS_FULL = (28, 142, 814, 538)
BBOX_NO_PERECEDEROS_FULL = (27, 42, 814, 296)

print("Configuracion cargada.")

## 1. Funciones de Extraccion

In [ ]:
def parse_header(texto_pagina):
    """Extrae metadata del boletin: numero, mes, ano, quincena."""
    m = re.search(
        r"N\u00ba\s*(\d+)\s+([A-Z\u00c1\u00c9\u00cd\u00d3\u00da]+)\s*-\s*(\d{4})\s+(PRIMERA|SEGUNDA)\s+QUINCENA",
        texto_pagina
    )
    if not m:
        return None
    num, mes, anio, quincena = m.groups()
    if mes not in MESES:
        return None
    return {
        "boletin_num": int(num),
        "mes": MESES[mes],
        "a\u00f1o": int(anio),
        "quincena": 1 if quincena == "PRIMERA" else 2,
        "quincena_id": f"{anio}-{MESES[mes]:02d}-Q{1 if quincena == 'PRIMERA' else 2}",
    }


def ocr_region(pagina_pdfplumber, bbox):
    """Renderiza una region de la pagina y aplica OCR.
    Fusiona bloques cercanos en Y para formar lineas completas."""
    recorte = pagina_pdfplumber.within_bbox(bbox)
    imagen = np.array(recorte.to_image(resolution=OCR_RESOLUTION).original)
    resultados, _ = OCR_ENGINE(imagen)

    if not resultados:
        return ""

    bloques = []
    for det in resultados:
        if not det or len(det) < 2:
            continue
        bbox_det, texto = det[0], str(det[1]).strip()
        if not texto:
            continue
        y_center = (bbox_det[0][1] + bbox_det[2][1]) / 2
        x_left = bbox_det[0][0]
        bloques.append((y_center, x_left, texto))

    bloques.sort(key=lambda b: (b[0], b[1]))

    UMBRAL_Y = 12
    lineas_fusionadas = []
    linea_actual = []
    y_ref = bloques[0][0]

    for y, x, texto in bloques:
        if abs(y - y_ref) > UMBRAL_Y:
            linea_actual.sort(key=lambda b: b[1])
            lineas_fusionadas.append(" ".join(b[2] for b in linea_actual))
            linea_actual = []
            y_ref = y
        linea_actual.append((y, x, texto))

    if linea_actual:
        linea_actual.sort(key=lambda b: b[1])
        lineas_fusionadas.append(" ".join(b[2] for b in linea_actual))

    return "\n".join(lineas_fusionadas)


def parsear_tabla_precios(texto_ocr):
    """Convierte texto OCR de una tabla de precios en registros, respetando el orden de lectura."""
    registros = []
    patron_precio = re.compile(r"^[\d]+(?:[.,]\d{1,2})?$")
    unidades_cantidad = {
        "lb", "l", "lt", "kg", "g", "gr", "ml", "cc",
        "c", "u", "und", "un", "funda", "malla", "caja",
        "bolsa", "paquete", "atajo", "ramo", "paca", "saco",
        "manojo", "botella", "frasco", "litro", "litros", "docena",
        "bandeja", "tarro", "bulto", "envase", "galon", "gal",
    }
    palabras_descartar = {
        "producto",
        "productos perecederos",
        "productos no perecederos",
        "fuente",
        "sistema",
        "usd/presentaci\u00f3n",
        "quincena",
        "boletin",
        "bolet\u00edn",
        "provincia",
        "precio",
        "tabla",
        "mes",
        "marzo",
        "abril",
        "mayo",
        "junio",
        "julio",
        "agosto",
        "septiembre",
        "octubre",
        "noviembre",
        "diciembre",
        "enero",
        "febrero",
        "azuay",
        "guayas",
        "pichincha",
        "at/t-1",
        "at/t",
        "a/t",
        "usd",
        "us$/presentacion",
        "us$/presentaci\u00f3n",
    }

    def to_float(v):
        if v is None:
            return None
        v = str(v).strip().rstrip("%")
        if not v or v == "-":
            return None
        v = v.replace(" ", "").replace(",", ".")
        if "." not in v and len(v) > 4 and v.isdigit():
            v = v[:-2] + "." + v[-2:]
        try:
            return float(v)
        except ValueError:
            return None

    for linea in texto_ocr.split("\n"):
        linea = linea.strip()
        if not linea:
            continue

        linea_limpia = re.sub(r"\s+", " ", linea.lower()).strip()
        if linea_limpia in palabras_descartar:
            continue
        if any(p in linea_limpia for p in palabras_descartar):
            continue
        if not re.search(r"[A-Za-zÁÉÍÓÚÑáéíóúñ]", linea):
            continue
        if re.fullmatch(r"[\d.,%/-]+", linea):
            continue

        tokens = linea.split()
        if not tokens:
            continue

        precio_tokens = []
        indices_precios = []
        variacion_raw = None
        for i in range(len(tokens) - 1, -1, -1):
            token_original = tokens[i]
            token = token_original.rstrip("%")
            if token_original.endswith("%") and variacion_raw is None:
                variacion_raw = token_original
                continue
            if token == "-":
                precio_tokens.insert(0, token)
                indices_precios.insert(0, i)
                if len(precio_tokens) == 2:
                    break
                continue
            if not patron_precio.match(token):
                continue
            if i + 1 < len(tokens) and re.search(r"[A-Za-zÁÉÍÓÚñ]", tokens[i + 1]):
                continue
            precio_tokens.insert(0, token)
            indices_precios.insert(0, i)
            if len(precio_tokens) == 2:
                break

        if len(precio_tokens) < 2:
            precio_tokens = []
            indices_precios = []

        if indices_precios:
            nombre_tokens = tokens[:indices_precios[0]]
        else:
            nombre_tokens = tokens

        nombre = " ".join(nombre_tokens).strip(" -\u2022")
        if not nombre:
            nombre = linea.strip(" -\u2022")
        if not re.search(r"[A-Za-zÁÉÍÓÚÑáéíóúñ]", nombre):
            continue

        nombre = re.sub(r"\bIb\b", "lb", nombre)
        nombre = re.sub(r"\blb\b", "lb", nombre)
        nombre = re.sub(r"\b1b\b", "lb", nombre)
        nombre = re.sub(r"\b\|b\b", "lb", nombre)
        nombre = re.sub(r"\bIt\b", "lt", nombre)
        nombre = re.sub(r"\b\|t\b", "lt", nombre)
        nombre = re.sub(r"\[", "(", nombre)
        nombre = re.sub(r"\(", "(", nombre)
        nombre = re.sub(r"\)", ")", nombre)
        nombre = re.sub(r"Invemadero", "Invernadero", nombre)
        nombre = re.sub(r"Tiema", "Tierna", nombre)
        nombre = re.sub(r"Tierma", "Tierna", nombre)
        nombre = re.sub(r"Tiera", "Tierna", nombre)
        nombre = re.sub(r"Fr[e\u00e9]jol", "Frejol", nombre)
        nombre = re.sub(r"aprox[_\s:]+", "aprox. ", nombre)
        nombre = re.sub(r"Se[n\u00f1]o", "Seco", nombre)
        nombre = re.sub(r"Se\x82o", "Seco", nombre)
        nombre = re.sub(r"Se\ufffd[o\u00f3]", "Seco", nombre)
        nombre = re.sub(r"\]$", ")", nombre)
        nombre = re.sub(r"Se[^c\d\s]{1,3}o(?=\s*\()", "Seco", nombre)
        nombre = re.sub(r"de 11\b", "de 1 l", nombre)
        nombre = re.sub(r"de 1 1\b", "de 1 l", nombre)
        nombre = re.sub(r"de 1 I\b", "de 1 l", nombre)
        nombre = re.sub(r"de 1I\)", "de 1 l)", nombre)
        nombre = re.sub(r"de 1l\)", "de 1 l)", nombre)
        nombre = re.sub(r"de 1 /\)", "de 1 l)", nombre)
        nombre = re.sub(r"de 1 \|\)", "de 1 l)", nombre)
        nombre = re.sub(r"de 1 \|", "de 1 l", nombre)
        nombre = re.sub(r"aprox\.\)", "aprox.)", nombre)
        nombre = re.sub(r"\(aprox\.\)", "", nombre)

        while nombre.startswith("(") and nombre.count("(") > nombre.count(")"):
            nombre = nombre[1:]
        nombre = re.sub(r"\)\)$", ")", nombre)
        if "(" in nombre and nombre.count("(") > nombre.count(")"):
            nombre = nombre + ")"
        nombre = re.sub(r"\s+\(\)$", "", nombre)
        nombre = re.sub(r"\(\)\s*\)", ")", nombre)
        nombre = re.sub(r"\(Envase de\)$", "", nombre)
        nombre = nombre.strip()

        if len(nombre) < 2:
            continue

        precio_anterior = to_float(precio_tokens[0]) if len(precio_tokens) >= 2 else None
        precio_actual = to_float(precio_tokens[1]) if len(precio_tokens) >= 2 else None

        variacion = None
        if variacion_raw:
            v = variacion_raw.rstrip("%").replace(" ", "")
            try:
                variacion = round(float(v), 1)
            except ValueError:
                variacion = None

        registros.append({
            "producto_raw": nombre,
            "precio_anterior": precio_anterior,
            "precio_actual": precio_actual,
            "variacion": variacion,
        })

    return registros


def dividir_en_provincias(bbox_full, n_provincias=3):
    x0, top, x1, bottom = bbox_full
    ancho_col = (x1 - x0) / n_provincias
    return [(x0 + i * ancho_col, top, x0 + (i + 1) * ancho_col, bottom) for i in range(n_provincias)]


def procesar_boletin(pdf_path):
    registros_totales = []
    encabezados_fallidos = []
    orden_global = 0

    with pdfplumber.open(pdf_path) as pdf:
        n_paginas = len(pdf.pages)

        for i in range(0, n_paginas, 2):
            pagina_portada = pdf.pages[i]
            texto_portada = pagina_portada.extract_text() or ""
            info = parse_header(texto_portada)

            if info is None:
                encabezados_fallidos.append(i + 1)
                print(f"  [!] No se pudo leer encabezado en pagina {i+1}, se omite.")
                continue

            print(f"  Procesando boletin No{info['boletin_num']} - {info['quincena_id']} ...")

            bboxes_perec = dividir_en_provincias(BBOX_PERECEDEROS_FULL)
            for provincia, bbox in zip(PROVINCIAS, bboxes_perec):
                texto = ocr_region(pagina_portada, bbox)
                filas = parsear_tabla_precios(texto)
                for f in filas:
                    f.update(info)
                    f["provincia"] = provincia
                    f["categoria"] = "perecedero"
                    f["orden"] = orden_global
                    orden_global += 1
                registros_totales.extend(filas)

            if i + 1 < n_paginas:
                pagina_2 = pdf.pages[i + 1]
                bboxes_noperec = dividir_en_provincias(BBOX_NO_PERECEDEROS_FULL)
                for provincia, bbox in zip(PROVINCIAS, bboxes_noperec):
                    texto = ocr_region(pagina_2, bbox)
                    filas = parsear_tabla_precios(texto)
                    for f in filas:
                        f.update(info)
                        f["provincia"] = provincia
                        f["categoria"] = "no_perecedero"
                        f["orden"] = orden_global
                        orden_global += 1
                    registros_totales.extend(filas)

    return registros_totales, encabezados_fallidos


def validar_calidad(df, encabezados_fallidos):
    problemas = []

    mask_precio_nulo = df["precio_anterior"].isna() | df["precio_actual"].isna()
    for _, row in df[mask_precio_nulo].iterrows():
        problemas.append({
            "tipo": "precio_nulo", "producto": row["producto_raw"],
            "provincia": row["provincia"], "quincena": row["quincena_id"],
            "detalle": "Precio anterior o actual es nulo"
        })

    mask_datos = df["producto_raw"].isna() | df["producto_raw"].str.len().fillna(0).eq(0)
    for _, row in df[mask_datos].iterrows():
        problemas.append({
            "tipo": "producto_vacio", "producto": row["producto_raw"],
            "provincia": row["provincia"], "quincena": row["quincena_id"],
            "detalle": "Nombre de producto vacio"
        })

    total = len(df)
    completitud = round(100 * (1 - len(problemas) / total), 2) if total > 0 else 0

    return {
        "total_registros": total,
        "registros_completos": total - len(problemas),
        "registros_con_problema": len(problemas),
        "porcentaje_completitud": completitud,
        "productos_unicos": df["producto_raw"].nunique(),
        "quincenas": df["quincena_id"].nunique(),
        "problemas": problemas,
        "encabezados_fallidos": encabezados_fallidos,
    }

## 2. Seleccionar PDF y Procesar

In [ ]:
PDF_PATH = "precios_mayoristas_2026.pdf"

print(f"Procesando: {PDF_PATH}")
registros, encabezados_fallidos = procesar_boletin(PDF_PATH)
print(f"\nTotal registros extraidos: {len(registros)}")

## 3. Resultados

In [ ]:
df = pd.DataFrame(registros)

if "orden" in df.columns:
    df = df.sort_values("orden").drop(columns=["orden"])

reporte = validar_calidad(df, encabezados_fallidos)

print("=" * 50)
print("REPORTE DE CALIDAD")
print("=" * 50)
print(f"Registros totales:     {reporte['total_registros']}")
print(f"Registros completos:   {reporte['registros_completos']} ({reporte['porcentaje_completitud']}%)")
print(f"Con problemas:         {reporte['registros_con_problema']}")
print(f"Productos unicos:      {reporte['productos_unicos']}")
print(f"Quincenas:             {reporte['quincenas']}")

tipos = {}
for p in reporte["problemas"]:
    tipos.setdefault(p["tipo"], []).append(p)

if tipos:
    print(f"\n--- PROBLEMAS ({len(reporte['problemas'])} total) ---")
    for tipo, items in tipos.items():
        print(f"\n  [{tipo.upper()}] ({len(items)} registros)")
        for e in items[:3]:
            print(f"    - {e['producto']} ({e['provincia']}, {e['quincena']})")
        if len(items) > 3:
            print(f"    ... y {len(items) - 3} mas")
print("=" * 50)

In [ ]:
df.head(20)

In [ ]:
CSV_PATH = "data/processed/dataset_crudo_sipa.csv"

os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Guardado: {CSV_PATH}")